[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Random Numbers &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


**1.** Make a generator with seed 7, roll ten dice with `integers`, and print the rolls and their mean.

In [2]:
rng = np.random.default_rng(7)
rolls = rng.integers(1, 7, size=10)

print(rolls, rolls.mean())

[6 4 5 6 4 5 6 2 1 2] 4.1


`high` is 7 so that 6 can appear.

**2.** With seed 8, draw 1,000 values from a normal distribution with mean 8 and standard deviation 6.73, and print their mean and standard deviation, rounded to two places.

In [3]:
rng = np.random.default_rng(8)
values = rng.normal(8, 6.73, size=1000)

print(values.mean().round(2), values.std().round(2))

7.81 6.83


A thousand draws come close to the mean and standard deviation asked for, and land on neither exactly.

**3.** With seed 9, pick three of the four station names at random, without replacement, using `choice`.

In [4]:
rng = np.random.default_rng(9)

print(rng.choice(list(STATIONS), size=3, replace=False))

['Svalbard' 'Tromso' 'Bergen']


`choice` takes a list as well as a number, and returns its picks as an array.

**4.** With seed 10, shuffle the positions of Oslo's 365 daily means with `permutation`, and print the mean of the first 73 shuffled days and the mean of all 365, rounded to two places.

In [5]:
rng = np.random.default_rng(10)
oslo_days = readings[1].reshape(365, 24).mean(axis=1)
shuffled = rng.permutation(365)

print(oslo_days[shuffled[:73]].mean().round(2), oslo_days.mean().round(2))

6.35 6.5


The 73 shuffled days come within a fraction of a degree of the year's mean.

**5.** With seed 12 and 10,000 random starting hours, estimate how often a 24-hour window at Tromso, row 3, is entirely below freezing, and compare it with the share among every window that fits in the year.

In [6]:
rng = np.random.default_rng(12)
window_starts = rng.integers(0, HOURS - 23, size=10_000)
windows = window_starts[:, np.newaxis] + np.arange(24)

print((readings[3][windows] < 0).all(axis=1).mean())

all_windows = np.arange(HOURS - 23)[:, np.newaxis] + np.arange(24)
print(round((readings[3][all_windows] < 0).all(axis=1).mean(), 4))

0.2219
0.2148


`all(axis=1)` asks whether every hour of a window was below freezing, where the worked example's `any` asked whether one was.

---

&#8592; **Back to:** [Random Numbers](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/12-random-numbers.ipynb)  &nbsp;&middot;&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
